# Arxiv Recommender GPU Workbench
대시보드에서 연결되는 GPU 학습·벤치마크 워크벤치입니다. **런타임 > 런타임 유형 변경 > GPU**를 선택한 뒤 위에서부터 실행하세요.

In [ ]:
import os, subprocess, sys, torch
assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다. 런타임 유형을 GPU로 변경하세요.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)

In [ ]:
REPOSITORY = 'https://github.com/YHFTF/arxiv-conversational-recommender.git' # @param {type:'string'}
BRANCH = 'main' # @param {type:'string'}
PROJECT_ROOT = '/content/arxiv-recsys'
if os.path.exists(os.path.join(PROJECT_ROOT, '.git')):
    subprocess.run(['git', '-C', PROJECT_ROOT, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', PROJECT_ROOT, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', PROJECT_ROOT, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPOSITORY, PROJECT_ROOT], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ARCHIVE = '/content/drive/MyDrive/colab_data.zip' # @param {type:'string'}
if os.path.exists(DATA_ARCHIVE):
    subprocess.run(['unzip', '-q', '-o', DATA_ARCHIVE, '-d', PROJECT_ROOT], check=True)
required = ['subdataset/build_hetero_graph_v2.pt', 'subdataset/arxiv_master_final.json', 'output/knowledge_meta.json']
missing = [path for path in required if not os.path.exists(os.path.join(PROJECT_ROOT, path))]
assert not missing, f'필수 데이터가 없습니다: {missing}. {DATA_ARCHIVE}를 확인하세요.'
print('데이터 준비 완료')

In [ ]:
TASK = 'train_v4' # @param ['train_v4', 'benchmark_v2', 'inference_v4']
INFERENCE_QUERY = 'graph neural networks' # @param {type:'string'}
commands = {
    'train_v4': [sys.executable, 'code/model/train_v4_knowledge_bpr.py'],
    'benchmark_v2': [sys.executable, 'code/test/run_benchmark_v2.py'],
    'inference_v4': [sys.executable, 'code/model/inference_v4.py', INFERENCE_QUERY],
}
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
print('실행:', ' '.join(commands[TASK]))
subprocess.run(commands[TASK], cwd=PROJECT_ROOT, env=env, check=True)

In [ ]:
import shutil, datetime
BACKUP_ROOT = '/content/drive/MyDrive/arxiv-recsys-results' # @param {type:'string'}
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
destination = os.path.join(BACKUP_ROOT, stamp)
os.makedirs(destination, exist_ok=True)
for path in ['output/lightgcn_v4_knowledge_bpr.pt', 'output/benchmark']:
    source = os.path.join(PROJECT_ROOT, path)
    if os.path.isdir(source): shutil.copytree(source, os.path.join(destination, os.path.basename(source)), dirs_exist_ok=True)
    elif os.path.isfile(source): shutil.copy2(source, destination)
print('결과 백업:', destination)